In [1]:
import time
import os
import numpy as np
from pathlib import Path
from gpytoolbox import remesh_botsch
import torch
from types import SimpleNamespace
import sys
from functools import reduce
from itertools import product
from typing import Tuple
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import warnings
import shutil
import random

sys.path.append("../flare") 
from flame.FLAME import FLAME
from flare.core import (
    Mesh, Renderer
)
from flare.losses import *
from flare.modules import (
    NeuralShader, get_deformer_network, Displacement
)
from flare.utils import (
    AABB, read_mesh, write_mesh,
    visualize_training,
    make_dirs, 
    set_defaults_finetune,
    save_individual_img,
    save_relit_intrinsic_materials
)
import nvdiffrec.render.light as light
from flare.dataset import DatasetLoader, dataset_util
from flare.dataset import *
from flare.metrics import metrics
import nvdiffrec.render.light as light

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/home/hleonhard/miniconda3/envs/flare/lib/python3.9/site-packages/torch/utils/cpp_extension.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


In [2]:
# ==============================================================================================
# evaluation
# ==============================================================================================    
def run(args, mesh, views, FLAMEServer, deformer_net, shader, renderer, device, channels_gbuffer, lgt):
    ## ============== deform ==============================     
    shapedirs, posedirs, lbs_weights = deformer_net.query_weights(mesh.vertices)
    eval_vertices = mesh.vertices
    batched_verts = eval_vertices.unsqueeze(0).repeat(views["img"].shape[0], 1, 1)

    _, pose_features, transformations = FLAMEServer(expression_params=views["flame_expression"], full_pose=views["flame_pose"])
    if args.ghostbone:
        transformations = torch.cat([torch.eye(4).unsqueeze(0).unsqueeze(0).expand(views["img"].shape[0], -1, -1, -1).float().to(device), transformations], 1)
    deformed_vertices = FLAMEServer.forward_pts_batch(pnts_c=batched_verts, betas=views["flame_expression"], transformations=transformations, pose_feature=pose_features, 
                                        shapedirs=shapedirs, posedirs=posedirs, lbs_weights=lbs_weights, dtype=torch.float32, map2_flame_original=True)

    d_normals = mesh.fetch_all_normals(deformed_vertices, mesh)
    ## ============== Rasterize ==============================
    gbuffers = renderer.render_batch(views["camera"], deformed_vertices.contiguous(), d_normals,
                        channels=channels_gbuffer, with_antialiasing=True, 
                        canonical_v=mesh.vertices, canonical_idx=mesh.indices)
    
    ## ============== predict color ==============================
    rgb_pred, cbuffers, gbuffer_mask = shader.shade(gbuffers, views, mesh, args.finetune_color, lgt)

    return rgb_pred, gbuffers, cbuffers

# ==============================================================================================
# relight: run
# ==============================================================================================  
def run_relight(args, mesh, views, FLAMEServer, deformer_net, shader, renderer, device, channels_gbuffer, lgt_list, images_save_path):
    ## ============== deform ==============================     
    shapedirs, posedirs, lbs_weights = deformer_net.query_weights(mesh.vertices)
    eval_vertices = mesh.vertices
    batched_verts = eval_vertices.unsqueeze(0).repeat(views["img"].shape[0], 1, 1)

    _, pose_features, transformations = FLAMEServer(expression_params=views["flame_expression"], full_pose=views["flame_pose"])
    if args.ghostbone:
        transformations = torch.cat([torch.eye(4).unsqueeze(0).unsqueeze(0).expand(views["img"].shape[0], -1, -1, -1).float().to(device), transformations], 1)
    deformed_vertices = FLAMEServer.forward_pts_batch(pnts_c=batched_verts, betas=views["flame_expression"], transformations=transformations, pose_feature=pose_features, 
                                        shapedirs=shapedirs, posedirs=posedirs, lbs_weights=lbs_weights, dtype=torch.float32, map2_flame_original=True)

    d_normals = mesh.fetch_all_normals(deformed_vertices, mesh)
    ## ============== Rasterize ==============================
    gbuffers = renderer.render_batch(views["camera"], deformed_vertices.contiguous(), d_normals,
                        channels=channels_gbuffer, with_antialiasing=True, 
                        canonical_v=mesh.vertices, canonical_idx=mesh.indices)
    
    ## ============== predict color ==============================
    relit_imgs, cbuffers, gbuffer_mask = shader.relight(gbuffers, views, mesh, args.finetune_color, lgt_list)
    save_relit_intrinsic_materials(relit_imgs, views, gbuffer_mask, cbuffers, images_save_path)

# ==============================================================================================
# evaluation: numbers
# ==============================================================================================  
def quantitative_eval(args, mesh, dataloader_validate, FLAMEServer, deformer_net, shader, renderer, device, channels_gbuffer,
                        experiment_dir, images_eval_save_path, lgt=None, save_each=False):

    for it, views_subset in enumerate(dataloader_validate):
        with torch.no_grad():
            rgb_pred, gbuffer, cbuffer = run(args, mesh, views_subset, FLAMEServer, deformer_net, shader, renderer, device, 
                    channels_gbuffer, lgt=lgt)

        rgb_pred = rgb_pred * gbuffer["mask"]
        if save_each:
            save_individual_img(rgb_pred, views_subset, gbuffer["normal"], gbuffer["mask"], cbuffer, images_eval_save_path)

    ## ============== metrics ==============================
    gt_dir = Path(args.input_dir)
    if gt_dir is not None:
        eval_list = metrics.run(images_eval_save_path, gt_dir, args.eval_dir)

    with open(str(experiment_dir / "final_eval.txt"), 'a') as f:
        f.writelines("\n"+"w/o cloth result:"+"\n")
        f.writelines("\n"+"MAE | LPIPS | SSIM | PSNR"+"\n")
        if gt_dir is not None:
            eval_list = [str(e) for e in eval_list]
            f.writelines(" ".join(eval_list))

In [10]:
config = {
    'config': None,
    'run_name': 'diff_renderer_0_0_0_0',
    'batch_size': 2,
    # path
    'input_dir': Path("/home/hleonhard/data/flare_subject_data/001"),
    'train_dir': ["MVI_1814", "MVI_1810"],
    'eval_dir': ["MVI_1812"],
    'working_dir': Path("/home/hleonhard/data/flare_train_setup"),
    'output_dir': Path("out"),
    # misc
    'sample_idx_ratio': 1,
    'device': 0,
    'finetune_color': False,
    # iters
    'iterations': 2000, 
    'final_iter': 1500,
    'upsample_iterations': [500],
    'save_frequency': 300,
    'visualization_frequency': 100,
    'visualization_views': [15, 25, 27, 21, 26],
    # 'downsample' default is set via set_defaults
    'downsample': False,
    'downsample_ratio': 0.03,
    'grad_scale': False, # Default for action='store_true' is False unless set otherwise
    # flame
    'decay_flame': [100],
    'flame_mask': False,
    # lr
    'lr_vertices': 1e-3,
    'lr_shader': 1e-3,
    'lr_deformer': 1e-3,
    # loss weights
    'weight_mask': 2.0,
    'weight_normal': 0.1,
    'weight_laplacian': 60.0,
    'weight_shading': 1.0,
    'weight_perceptual_loss': 0.1,
    'weight_albedo_regularization': 0.01,
    'weight_flame_regularization': 10.0,
    'weight_white_lgt_regularization': 1.0,
    'weight_roughness_regularization': 0.1,
    'weight_fresnel_coeff': 0.01,
    # diffusion regularization
    'diffusion_dir': Path("/home/hleonhard/data/flare_diffusion_channels/rgbx/001"),
    "diffusion_normal": 0.0,
    "diffusion_albedo": 0.0,
    "diffusion_roughness": 0.0,
    "diffusion_irradiance": 0.0,
    'r_mean': 0.500,
    # neural shader
    'fourier_features': 'positional',
    'activation': 'relu',
    'bsdf': 'pbr_shading',
    'deform_d_out': 128,
    'light_mlp_ch': 3,
    'light_mlp_dims': [64, 64],
    'material_mlp_dims': [128, 128, 128, 128, 128],
    'material_mlp_ch': 5,
    # ghostbone/train_deformer (defaults are set via set_defaults)
    'ghostbone': True,
    'train_deformer': True,
    'deform_dims': [128, 128, 128, 128]
}
args = SimpleNamespace(**config)

In [11]:
device = torch.device('cpu')
if torch.cuda.is_available() and args.device >= 0:
    device = torch.device(f'cuda:{args.device}')
print(f"Using device {device}")

# Create directories
run_name = args.run_name if args.run_name is not None else args.input_dir.parent.name
images_save_path, images_eval_save_path, meshes_save_path, shaders_save_path, experiment_dir = make_dirs(args, run_name, args.finetune_color)
flame_path = "/home/hleonhard/adl4cv_ws25-26_Relightable-Avatars/flare/flame/FLAME2020/generic_model.pkl"

Using device cuda:0


In [12]:
images_save_path, images_eval_save_path, meshes_save_path, shaders_save_path, experiment_dir

(PosixPath('/home/hleonhard/data/flare_train_setup/out/diff_renderer_0_0_0_0/stage_1/images'),
 PosixPath('/home/hleonhard/data/flare_train_setup/out/diff_renderer_0_0_0_0/images_evaluation'),
 PosixPath('/home/hleonhard/data/flare_train_setup/out/diff_renderer_0_0_0_0/stage_1/meshes'),
 PosixPath('/home/hleonhard/data/flare_train_setup/out/diff_renderer_0_0_0_0/stage_1/network_weights'),
 PosixPath('/home/hleonhard/data/flare_train_setup/out/diff_renderer_0_0_0_0'))

In [13]:
### Read the views
print("loading test views...")
dataset_val      = DatasetLoader(args, train_dir=args.eval_dir, sample_ratio=args.sample_idx_ratio, pre_load=True)
dataloader_validate = torch.utils.data.DataLoader(dataset_val, batch_size=4, collate_fn=dataset_val.collate, shuffle=False)
### init flame and deformation
flame_shape = dataset_val.shape_params
FLAMEServer = FLAME(flame_path, n_shape=100, n_exp=50, shape_params=flame_shape).to(device)
### Obtain the initial mesh and compute its connectivity
flame_canonical_mesh = Mesh(FLAMEServer.v_template, FLAMEServer.faces_tensor, device=device)
flame_canonical_mesh.compute_connectivity()
### create bounding box from the mesh vertices
aabb = AABB(flame_canonical_mesh.vertices.cpu().numpy())
flame_mesh_aabb = [torch.min(flame_canonical_mesh.vertices, dim=0).values, torch.max(flame_canonical_mesh.vertices, dim=0).values]
# init mesh is mouth open!!!
FLAMEServer.canonical_exp = dataset_val.get_mean_expression_train(args.train_dir).to(device)
FLAMEServer.canonical_pose = FLAMEServer.canonical_pose.to(device)
FLAMEServer.canonical_verts, FLAMEServer.canonical_pose_feature, FLAMEServer.canonical_transformations = \
    FLAMEServer(expression_params=FLAMEServer.canonical_exp, full_pose=FLAMEServer.canonical_pose)
FLAMEServer.canonical_verts = FLAMEServer.canonical_verts.to(device)
flame_canonical_mesh.vertices = FLAMEServer.canonical_verts.squeeze(0)
# ==============================================================================================
# mesh
# ==============================================================================================

mesh_path = Path(experiment_dir / "stage_2" / "meshes" / f"mesh_latest.obj")
mesh = read_mesh(mesh_path, device=device)
mesh.compute_connectivity()
mesh.to(device)
print("loaded mesh")
# ==============================================================================================
# Rendererrr
# ==============================================================================================
renderer = Renderer(device=device)
renderer.set_near_far(dataset_val, torch.from_numpy(aabb.corners).to(device), epsilon=0.5)
channels_gbuffer = ['mask', 'position', 'normal', "canonical_position"]
print("Rasterizing:", channels_gbuffer)
# ==============================================================================================
# deformation 
# ==============================================================================================
load_deformer = Path(experiment_dir / "stage_2" / "network_weights" / f"deformer_latest.pt")
assert os.path.exists(load_deformer)
multires = 0
deformer_net = get_deformer_network(FLAMEServer, model_path=load_deformer, train=False, d_in=3, dims=[128, 128, 128, 128], 
                                       weight_norm=True, multires=multires, num_exp=50, aabb=aabb, ghostbone=args.ghostbone, device=device)
if args.ghostbone:
    FLAMEServer.canonical_transformations = torch.cat([torch.eye(4).unsqueeze(0).unsqueeze(0).float().to(device), FLAMEServer.canonical_transformations], 1)
# ==============================================================================================
# shading
# ==============================================================================================
load_shader = Path(experiment_dir / "stage_2" / "network_weights" / f"shader_latest.pt")
assert os.path.exists(load_shader)
shader = NeuralShader.load(load_shader, device=device)
lgt = light.create_env_rnd()    
print("=="*50)
shader.eval()
deformer_net.eval()
batch_size = args.batch_size
print("Batch Size:", batch_size)

loading test views...
loaded 1824 views
creating the FLAME Decoder
loaded mesh
Rasterizing: ['mask', 'position', 'normal', 'canonical_position']
STAGE 2: Using hashgrid (tinycudann) for intrinsic materials
{'material_mlp_ch': 5, 'light_mlp_ch': 3, 'material_mlp_dims': [64, 64], 'light_mlp_dims': [64, 64]}
Batch Size: 2


In [14]:
 # ==============================================================================================
# evaluation: intrinsic materials and relighting
# ==============================================================================================  
lgt_list = light.load_target_cubemaps(Path("/home/hleonhard/adl4cv_ws25-26_Relightable-Avatars/flare/"))
for i in range(len(lgt_list)):
    Path(images_eval_save_path / "qualitative_results" / f"env_map_{i}" ).mkdir(parents=True, exist_ok=True)
lgt_list

loading the following environment maps:
/home/hleonhard/adl4cv_ws25-26_Relightable-Avatars/flare/assets/env_maps/arboretum_1k.hdr
/home/hleonhard/adl4cv_ws25-26_Relightable-Avatars/flare/assets/env_maps/solitude_night_1k.hdr
/home/hleonhard/adl4cv_ws25-26_Relightable-Avatars/flare/assets/env_maps/the_sky_is_on_fire_1k.hdr


[EnvironmentLight(), EnvironmentLight(), EnvironmentLight()]

In [15]:
images_eval_save_path

PosixPath('/home/hleonhard/data/flare_train_setup/out/diff_renderer_0_0_0_0/images_evaluation')

In [16]:
for it, views_subset in enumerate(dataloader_validate):
    with torch.no_grad():
        run_relight(args, mesh, views_subset, FLAMEServer, deformer_net, shader, renderer, device, channels_gbuffer, lgt_list, images_eval_save_path / "qualitative_results")